# Treino do detector de buracos (YOLO nano → ONNX para o navegador)

Notebook para o **Google Colab com GPU gratuita**: *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*.

Etapas:
1. Preparar o ambiente (clonar o repositório e instalar as dependências)
2. Baixar o dataset público de buracos (Kaggle, sem chave; ou Roboflow, com chave)
3. *(opcional)* Acrescentar as fotos próprias de calçadas anotadas no Roboflow
4. Combinar tudo nas nossas classes (`buraco`; depois `obstaculo_suspenso`)
5. Fazer o fine-tuning do **yolo11n** em 320×320 e avaliar (mAP, precisão, recall)
6. Exportar para **ONNX** (opset 12, sem NMS) + `labels.json` e baixar

Por que **um único modelo só com as nossas classes**: no navegador, cada modelo é uma inferência inteira a mais por frame. Detalhes em `docs/decisoes.md`.

In [ ]:
!nvidia-smi -L

## 1. Ambiente

Se o repositório for **privado**, o `git clone` falha. Nesse caso, faça upload da pasta `training/` compactada (`training.zip`) pelo painel de arquivos e rode a célula com `USAR_UPLOAD = True`.

In [ ]:
REPO = "https://github.com/kauannew/Camera_auxiliar_DV.git"
USAR_UPLOAD = False   # True se você subiu training.zip manualmente

import os
if USAR_UPLOAD:
    !unzip -q -o /content/training.zip -d /content/Camera_auxiliar_DV
elif not os.path.exists("/content/Camera_auxiliar_DV"):
    !git clone -q $REPO /content/Camera_auxiliar_DV

%cd /content/Camera_auxiliar_DV/training
!pip install -q -r requirements.txt

## 2. Dataset público de buracos

**Opção A (padrão, sem chave):** dataset de buracos do Kaggle `andrewmvd/pothole-detection` (anotações Pascal VOC, classe `pothole`). O `kagglehub` baixa datasets públicos sem login.

**Opção B (Roboflow Universe):** escolha um projeto de buracos em formato de detecção, confira as imagens e a licença, e preencha workspace/projeto/versão. A chave vai nos *Secrets* do Colab (ícone de chave na barra lateral) com o nome `ROBOFLOW_API_KEY`. Nunca deixe a chave escrita no notebook.

⚠️ Datasets públicos mostram principalmente buracos **de rua vistos do carro**. Para o nosso uso (calçada, celular no peito), as fotos próprias da etapa 3 fazem muita diferença.

In [ ]:
FONTE = "kaggle"   # "kaggle" ou "roboflow"

DESTINO = "datasets/brutos/publico"
!rm -rf $DESTINO

if FONTE == "kaggle":
    import kagglehub, shutil
    baixado = kagglehub.dataset_download("andrewmvd/pothole-detection")
    shutil.copytree(baixado, DESTINO)
    FORMATO_PUBLICO = "voc"   # esse dataset vem em Pascal VOC (.xml)
else:
    RF_WORKSPACE = "preencha"     # ex.: o trecho da URL universe.roboflow.com/<workspace>/<projeto>
    RF_PROJETO   = "preencha"
    RF_VERSAO    = 1
    from roboflow import Roboflow
    try:
        from google.colab import userdata
        chave = userdata.get("ROBOFLOW_API_KEY")
    except Exception:
        from getpass import getpass
        chave = getpass("Chave da API do Roboflow: ")
    Roboflow(api_key=chave).workspace(RF_WORKSPACE).project(RF_PROJETO) \
        .version(RF_VERSAO).download("yolov8", location=DESTINO)
    FORMATO_PUBLICO = "yolo"

!find $DESTINO -maxdepth 2 | head -20

In [ ]:
# Ajusta o formato da fonte "publico" em config/dataset.yaml conforme a opção escolhida
import yaml
cfg = yaml.safe_load(open("config/dataset.yaml", encoding="utf-8"))
for f in cfg["fontes"]:
    if f["nome"] == "publico":
        f["formato"] = FORMATO_PUBLICO
yaml.safe_dump(cfg, open("config/dataset.yaml", "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
print(cfg["fontes"])

## 3. *(Opcional, recomendado)* Fotos próprias de calçadas

1. Com o celular no suporte de peito, grave vídeos andando por calçadas com buracos e extraia ~1 frame por segundo (ou tire fotos).
2. Suba no Roboflow, anote a classe `buraco` (e, na fase 2, `obstaculo_suspenso`) e gere uma versão.
3. Baixe abaixo e **descomente** a fonte `calcadas_recife` em `config/dataset.yaml`.

In [ ]:
BAIXAR_PROPRIO = False
if BAIXAR_PROPRIO:
    from roboflow import Roboflow
    from google.colab import userdata
    rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
    rf.workspace("SEU_WORKSPACE").project("calcadas-recife").version(1) \
        .download("yolov8", location="datasets/brutos/calcadas_recife")

## 4. Combinar no dataset final

Remapeia as classes das fontes (`pothole` → `buraco`), converte VOC/segmentação em bbox YOLO e divide em train/val/test.

In [ ]:
!python scripts/preparar_dataset.py --config config/dataset.yaml

## 5. Fine-tuning do yolo11n em 320×320

Treinamos direto na resolução de uso no celular (320). Na T4, 80 épocas levam de alguns minutos a algumas dezenas de minutos, conforme o tamanho do dataset. O `patience` interrompe se a validação estagnar.

In [ ]:
!python scripts/treinar.py --data datasets/calcada/data.yaml --modelo yolo11n.pt --epochs 80 --batch 32 --device 0

In [ ]:
# Curvas de treino e exemplos de predição no conjunto de validação
from IPython.display import Image, display
import glob
pasta = "runs/buraco_yolo11n_320"
for nome in ["results.png", "confusion_matrix_normalized.png", "val_batch0_pred.jpg"]:
    for arq in glob.glob(f"{pasta}/{nome}"):
        print(arq); display(Image(arq, width=900))

### Leitura das métricas (para o pitch)
- **Recall** é a métrica mais importante para segurança: buraco não detectado = alerta perdido.
- **Precisão** baixa = alarmes falsos. O app compensa em parte exigindo **N frames consecutivos** antes de alertar.
- **mAP50** resume as duas coisas. Para um MVP de hackathon, algo acima de ~0,6 no teste já rende uma demo convincente.

## 6. Export para ONNX + labels.json

In [ ]:
!python scripts/exportar_onnx.py --pesos runs/buraco_yolo11n_320/weights/best.pt --nome buraco --sem-copiar
# Confere o ONNX com o mesmo pós-processamento do app e mede o tempo em CPU (1 thread)
amostra = sorted(glob.glob("datasets/calcada/images/test/*"))[0]
!python scripts/verificar_onnx.py --modelo exportados/buraco.onnx --imagem "$amostra" --bench 30

In [ ]:
# Baixa o modelo, os rótulos e as métricas. Coloque os dois primeiros em web/public/modelos/
!cd exportados && zip -q -j /content/modelo_buraco.zip buraco.onnx buraco.labels.json \
    ../runs/buraco_yolo11n_320/results.csv ../runs/buraco_yolo11n_320/results.png
from google.colab import files
files.download("/content/modelo_buraco.zip")